# Library

In [64]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Python path
import sys
base_folder = 'DataScience'
location_base = os.path.join(os.getcwd().split(base_folder)[0], base_folder)
location_module = [os.path.join(location_base, 'Module')] 
for each in location_module:
    if each not in sys.path:
        sys.path.append(each)

# Auto reload of library
%reload_ext autoreload
%autoreload 2

# Device Strategy
from import_KK import *
strategy, torch_device, num_gpus = DeviceStrategy()
from preprocessing_project_KK import *
from preprocessing_KK import *
from description_KK import *
from algorithm_machinelearning_KK import *
from algorithm_deeplearning_KK import *
from algorithm_causalml_KK import *
from evaluation_KK import *
from visualization_KK import *


=========== Device Strategy ===========
Detected GPUs:
 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
TF isolated to CPU (All 24GB VRAM reserved exclusively for PyTorch/XGBoost)
======= GPU / CUDA / STATUS ========
Cuda Ready?         True
CUDA Version:       12.5.1
cuDNN Version:      9

TF Version:         2.20.0
Keras Version:      3.15.1

------- Torch Check ----------------
Torch CUDA Available: True
Torch Version:        2.14.0+cu130
Torch CUDA Version:   13.0
Torch cuDNN Version:  92501
Torch GPUs Available: 2
Use the GPU [0]:    NVIDIA GeForce RTX 3090 (Index: 0)
Use the GPU [1]:    NVIDIA GeForce RTX 3090 (Index: 1)



# Hyperparameters

In [55]:
# Data
# FOLDER_GDM, FOLDER_GDS = get_google_drivelocation(gdrive='z', glocation=os.path.join('Research', 'SavedData', 'NTIS'))
# FOLDER_LOCATION = FOLDER_GDM
FOLDER_LOCATION = os.path.join('.', 'Data')
FILENAME = 'ebitda_margin_delta_pre_split_2016_2025.csv'
FILE_LOCATION = os.path.join(FOLDER_LOCATION, FILENAME)
LANGUAGE = 'en'

# Data Filtering
YEAR_MIN = 2016
YEAR_MAX = 2025
AGE_MIN = None
AGE_MAX = None
GENDER = 'all' # 'male', 'female'

# Preprocessing
Y_colname = 'y_delta_EBITDA대매출액'
Y_candidate = None
Time_colname = "target_year"
T_colname = None
X_reverse = None
X_encoding = None
X_dummy = None
X_moderate = None
X_control = None
X_delYrelated = None
X_del = None
X_delete = (X_delYrelated or []) + (X_del or [])
X_nullmax = None
X_nullmin = None
X_handle_nonres = False
X_nonres_col = None
WINSORIZE_YLIMITS = (0.01, 0.01)
# WINSORIZE_YLIMITS = None
# WINSORIZE_XLIMITS = (0.01, 0.01)
WINSORIZE_XLIMITS = None

TEST_SIZE = 'Time'
YEAR_VAL, YEAR_TEST = 1, 1 # Execute when TEST_SIZE = 'Time'
IMPUTE_METHOD = 'mice' # 'knn', 'mice', 'median'
RANDOM_STATE = 123
SAMPLING_METHOD = None    # 'auto'
# LABEL_LIST = ['Non-Violence', 'Violence']
LABEL_LIST = None

# Model
ALGO_NAMES_ML = ['Linear Regression', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
# ALGO_NAMES_ML = ['LightGBM', 'CatBoost']
ALGO_NAMES_DL = ['MLP', 'CNN', 'TabNet']
# ALGO_NAMES_DL = ['TabNet']
## Common
TUNE_TRIALS_ML = 100
TUNE_TRIALS_DL = 50
CV_SPLITS = 5
TUNE_METRIC_ML = 'RMSE' # Default('auto'): 'F1-score', 'RMSE' - 접두사 없어도 자동으로 validation set 기준으로 판단TUNE_METRIC = 'auto'
TUNE_METRIC_DL = 'val_RMSE' # 'AUC', 'Accuracy', 'F1-score', 'loss', 'val_AUC', 'val_Accuracy', 'val_F1-score', 'val_loss'
METRIC_FINAL = "RMSE"
## DL
CLASS_WEIGHT_METHOD = 'class' # class, sample, user_defined_dict
tf.random.set_seed(123)
NODE_MLP = [256, 512, 256, 128, 64]
NODE_CNN = [128, 256, 512]
VALIDATION_SPLIT = 0.2
if VALIDATION_SPLIT != None:
    VALIDATION_DATA = None
BATCH_SIZE = 1024
EPOCHS = 100
VERBOSE = 0
## XAI
SHAP = True
INTERACTION = False
INTERACTION_SAMPLES = 5000
INTERACTION_BATCH = 100
X_TOP_DISPLAY = 50
## Causal Inference
if T_colname != None:
    CAUSAL_CRITERION = 'het'
    CAUSAL_HONEST = True
    CAUSAL_INFERENCE = True
CV_CANDIDATE = [3,5,10]
N_ESTIMATORS = 1000
ROBUST_SAMPLES = 10000
CFSHAP_SAMPLES = 3000
CFSHAP_BATCH = 100

# Save
SAVE_NAME_PREDSTATTR = 'DescriptiveStatistics_BinaryPredTrain.csv'
SAVE_NAME_PREDSTATTE = 'DescriptiveStatistics_BinaryPredTest.csv'
SAVE_NAME_ML = 'Performance_ML'
SAVE_NAME_DL = 'Performance_DL'
SAVE_NAME_PERFORMANCE = 'Performance.csv'
SAVE_FOLDER_BASE = os.path.join('.', 'Result', 'Base')
SAVE_FOLDER_TEST = os.path.join('.', 'Result', 'Test')

In [48]:
def apply_missing_domain_rules(df):
    TARGET_YEAR_COLUMN = "target_year"
    TARGET_COLUMN = "y_delta_EBITDA대매출액"
    FEATURE_YEAR_COLUMN="feature_year"
    GRADE_COLUMNS = ["ESG등급", "환경등급", "사회등급", "지배구조등급"]
    GRADE_SCORE = {"D": 1.0, "C": 2.0, "B": 3.0, "B+": 4.0, "A": 5.0, "A+": 6.0}
    ALLOWANCE_MARKET_COLUMN = "배출권_시장총수량"
    ALLOWANCE_PARTICIPATION_COLUMN = "배출권_참여여부"

    STOCK_COLUMN = "종목코드"
    NEXT_MARGIN_COLUMN = "EBITDA대매출액_t1"
    TARGET_COLUMN = "y_delta_EBITDA대매출액"

    data = df.copy()
    data[TARGET_YEAR_COLUMN] = pd.to_numeric(data[TARGET_YEAR_COLUMN]).astype(int)
    data[FEATURE_YEAR_COLUMN] = pd.to_numeric(data[FEATURE_YEAR_COLUMN]).astype(int)
    data[TARGET_COLUMN] = pd.to_numeric(data[TARGET_COLUMN])
    for column in data.select_dtypes(include=["object", "string"]).columns:
        data[column] = data[column].astype("string").str.strip()

    allowance_source_columns = [
        column
        for column in data.columns
        if column.startswith("배출권_") and column != ALLOWANCE_MARKET_COLUMN
    ]
    allowance_zero_columns = [
        column
        for column in data.columns
        if "배출권" in column and column != ALLOWANCE_MARKET_COLUMN
    ]
    data[allowance_zero_columns] = data[allowance_zero_columns].apply(pd.to_numeric)
    data[ALLOWANCE_PARTICIPATION_COLUMN] = (
        data[allowance_source_columns].notna().any(axis=1).astype(int)
    )
    data[allowance_zero_columns] = data[allowance_zero_columns].fillna(0.0)

    # 사용자 확인: 해당 두 연도의 무거래 경매가격 결측은 0이다.
    auction_column = "경매_평균가격_원_tCO2eq"
    no_auction_years = data[FEATURE_YEAR_COLUMN].isin([2015, 2017])
    data.loc[no_auction_years, auction_column] = data.loc[
        no_auction_years, auction_column
    ].fillna(0.0)

    ordered = data.sort_values([STOCK_COLUMN, FEATURE_YEAR_COLUMN]).copy()
    for column in GRADE_COLUMNS:
        raw_grade = ordered[column]
        valid_grade = raw_grade.where(raw_grade.isin(GRADE_SCORE))
        previous_grade = valid_grade.groupby(ordered[STOCK_COLUMN], sort=False).ffill()
        # 보완된 값이 아닌 원래 유효등급의 관측연도로 유효기간을 계산한다.
        previous_grade_year = (
            ordered[FEATURE_YEAR_COLUMN].where(valid_grade.notna())
            .groupby(ordered[STOCK_COLUMN], sort=False).ffill()
        )
        unpublished = raw_grade.eq("미공개").fillna(False)
        needs_fill = unpublished | raw_grade.isna()
        grade_age = ordered[FEATURE_YEAR_COLUMN] - previous_grade_year
        eligible = previous_grade.notna() & grade_age.between(0, 4)
        fill_mask = needs_fill & eligible
        filled_grade = raw_grade.copy()
        filled_grade.loc[needs_fill] = pd.NA
        filled_grade.loc[fill_mask] = previous_grade.loc[fill_mask]
        ordered[column] = filled_grade.map(GRADE_SCORE).astype(float)
    for column in GRADE_COLUMNS:
        data[column] = ordered[column].reindex(data.index).astype(float)

    data = data.sort_values([TARGET_YEAR_COLUMN, STOCK_COLUMN]).reset_index(drop=True)
    return data

def preprocessing_CarbonFinance(
    df_raw,
    Y_colname='y_delta_EBITDA대매출액',
    firm_id_col='종목코드',
    feature_year_col='feature_year',
    target_year_col='target_year',
    X_delete=None,
):
    start_time = time.time()
    df = df_raw.copy()
    print("\n" + "="*65)
    print(" [PHASE 1] Preprocessing")
    print(f" Initial Shape: {df.shape}")
    print("="*65)

    # 1. 시계열 패널 정렬
    df = df.sort_values(by=[firm_id_col, feature_year_col]).reset_index(drop=True)
    print(f"▶ Sorted Panel: By [{firm_id_col}] & [{feature_year_col}]")

    # 2. 사용자 지정 컬럼 삭제
    if X_delete is not None:
        user_drop = [c for c in X_delete if c in df.columns]
        df = df.drop(columns=user_drop)
        print(f"▶ Dropped User-specified Columns: {len(user_drop)} columns")

    # 3. Duplicate columns and infinite numeric values are invalid inputs.
    if df.columns.duplicated().any():
        duplicate_columns = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate columns: {duplicate_columns}")

    num_cols = df.select_dtypes(include=[np.number]).columns
    if np.isinf(df[num_cols].to_numpy(dtype=float)).any():
        raise ValueError("Infinite values in numeric columns")

    YEAR_COLNAME = target_year_col

    # 4. 피처 블록 정의
    feature_cols = [c for c in df.columns if c not in [Y_colname, target_year_col]]

    b2_carbon = []
    b3_social = []
    b4_industry = []
    b1_financial = []

    # 탄소 및 배출권시장 키워드
    carbon_kws = ['Scope', 'scope', '배출량', '배출권', '할당량', '온실가스', '외부비용', '잠재탄소비용', 'tCO2e', '탄소', '경쟁', '협의', '경매', '장외', '소계']
    # 종합 ESG 및 인적자본 키워드
    social_kws = ['기부금', '사회등급', '지배구조등급', 'ESG', '종업원', '인건비', '급여', '근속', '복리후생', '환경등급', '노동생산성', '노동소득분배율']
    # 산업/거시/시장구분 핵심 키워드
    industry_kws = ['KSIC', 'Trucost산업', 'NICS', '주식업종', '시장구분', 'HHI', '점유율', '시장지배력']

    for col in feature_cols:
        # 1순위: 탄소 및 기후 리스크 블록
        if any(k in col for k in carbon_kws):
            b2_carbon.append(col)
        # 2순위: 종합 ESG 및 인적자본 블록
        elif any(k in col for k in social_kws):
            b3_social.append(col)
        # 3순위: 산업 고정효과 및 거시 경쟁 블록
        elif any(k in col for k in industry_kws):
            b4_industry.append(col)
        # 4순위: 나머지 모든 전통 재무, 손익, 대차, 현금흐름, 시장가치 및 결측 인디케이터
        else:
            b1_financial.append(col)

    feature_blocks = {
        'Block1_Financial_Control': b1_financial,
        'Block2_Carbon_Risk': b2_carbon,
        'Block3_Social_Governance': b3_social,
        'Block4_Industry_Macro': b4_industry
    }

    print("\n[Feature Block Curation Summary (MECE)]")
    total_assigned = 0
    for b_name, cols in feature_blocks.items():
        print(f"  ▷ {b_name:<26}: {len(cols):>4} features")
        total_assigned += len(cols)
    print(f"  ▷ Total Allocated / Total Available: {total_assigned} / {len(feature_cols)} (Orphan: {len(feature_cols) - total_assigned})\n")

    print(f"\nCompleted Domain Preprocessing in {time.time()-start_time:.2f}s")
    print(f"Final Data Shape: {df.shape} | Assigned YEAR_COLNAME: '{YEAR_COLNAME}'\n")

    return df.reset_index(drop=True), YEAR_COLNAME, feature_blocks

# Data Preprocessing

In [40]:
# Data Loading
df_raw = pd.read_csv(os.path.join(FOLDER_LOCATION, FILENAME))

In [59]:
# Preprocessing Data
df_missing_clean = apply_missing_domain_rules(df_raw)
df_prep, YEAR_COLNAME, FEATURE_BLOCKS = preprocessing_CarbonFinance(
    df_missing_clean,
    Y_colname=Y_colname,                
    X_delete=X_delete
)
# df_prep, YEAR_COLNAME, FEATURE_BLOCKS = preprocessing_CarbonFinance(
#     df_raw,
#     year_min=YEAR_MIN, 
#     year_max=YEAR_MAX,
#     Y_colname=Y_colname,          
#     drop_firm_dummies=True,        
#     create_lags=True,          
#     X_delete=X_delete
# )

# Preprocessing for Modeling
X_train, X_test, Y_train, Y_test, \
TASK, df_final, X_colname, \
CV_SPLITS, LABEL_LIST = preprocessing_modeling_KK(
    df_raw=df_prep, Y_colname=Y_colname, Time_colname=Time_colname, 
    X_reverse=X_reverse, X_encoding=X_encoding,
    X_dummy=X_dummy, X_moderate=X_moderate, X_control=X_control, X_delete=X_delete,
    winsorize_Ylimits=WINSORIZE_YLIMITS,
    winsorize_Xlimits=WINSORIZE_XLIMITS,
    test_size=TEST_SIZE, year_val=YEAR_VAL, year_test=YEAR_TEST,
    impute_method=IMPUTE_METHOD,
    sampling_method=SAMPLING_METHOD,
    label_list=LABEL_LIST,
)


 [PHASE 1] Preprocessing
 Initial Shape: (6087, 182)
▶ Sorted Panel: By [종목코드] & [feature_year]
▶ Dropped User-specified Columns: 0 columns

[Feature Block Curation Summary (MECE)]
  ▷ Block1_Financial_Control  :   77 features
  ▷ Block2_Carbon_Risk        :   63 features
  ▷ Block3_Social_Governance  :   27 features
  ▷ Block4_Industry_Macro     :   13 features
  ▷ Total Allocated / Total Available: 180 / 180 (Orphan: 0)


Completed Domain Preprocessing in 0.01s
Final Data Shape: (6087, 182) | Assigned YEAR_COLNAME: 'target_year'


Deleting Non-meaning & Data Leakage Variables...
Shape after dropping: (6087, 182)

▶ Detected Task Type: [ REGRESSION ] (Target: continuous)

Starting Preprocessing for REGRESSION...
Initial Shape: (6087, 182)

[OOT Split] Year Cutoff: Val @ 2024, Test @ 2025
▶ Target (Y) Winsorized (Train Quantiles): Low=-0.2316, High=0.2226

[Missing Handling (MICE)] Processing 126 columns...
    ▷ Completed in 28.42s (Remaining Train NaNs: 4778)

▶ Generating Regressio

,,Overall Sample,Target (Y) Subgroup,Statistics
Feature,Category / Metric,,,
EBITDA대매출액_t1,Mean ± SD,0.12 ± 0.12,-,"$r$=0.32, $p$<0.001"
fe_영업이익_매출,Mean ± SD,0.07 ± 0.13,-,"$r$=-0.31, $p$<0.001"
EBITDA대매출액,Mean ± SD,0.12 ± 0.12,-,"$r$=-0.31, $p$<0.001"
총자본순이익율,Mean ± SD,4.07 ± 8.68,-,"$r$=-0.21, $p$<0.001"
EBITDA대매출액__lag1,Mean ± SD,0.12 ± 0.13,-,"$r$=-0.20, $p$<0.001"
...,...,...,...,...
유형자산계_연결,Mean ± SD,1604586965286.98 ± 9210092387673.10,-,"$r$=0.00, $p$=0.97"
유형자산_투자부동산감가상각비_연결,Mean ± SD,177587000155.27 ± 1290504716060.50,-,"$r$=0.00, $p$=0.97"
fe_Scope1_2_매출집약도__diff1,Mean ± SD,0.06 ± 224.67,-,"$r$=0.00, $p$=0.98"



Final Shapes -> Train: (5473, 180), Test: (614, 180)
Total Preprocessing Time: 0:00:28
▶ Data Ready | CV Split Info: <class 'tuple'>



In [63]:
[col for col in df_prep.columns]

['target_year',
 'feature_year',
 '종목코드',
 'EBITDA대매출액_t1',
 'Trucost산업',
 '시장구분',
 'NICS산업분류',
 '주식업종',
 'KSIC대분류_11차',
 'KSIC중분류_11차',
 'KSIC소분류_11차',
 'KSIC세분류_11차',
 'Scope1직접배출량_tCO2e',
 'Scope1자료출처',
 'Scope1보고추정구분',
 'Scope2사업장간접배출량_tCO2e',
 'Scope2사업장간접자료출처',
 'Scope2사업장간접보고추정구분',
 'Scope3협력사간접배출량_tCO2e',
 'Scope3자료출처',
 'Scope3보고추정구분',
 '총매출_백만USD',
 '온실가스직접외부비용_백만USD',
 '매출대비온실가스직접외부비용비율_pct',
 '자산총계_연결',
 '유동자산계_연결',
 '비유동자산계_연결',
 '현금및현금성자산_연결',
 '매출채권_연결',
 '재고자산계_연결',
 '유형자산계_연결',
 '무형자산계_연결',
 '부채총계_연결',
 '유동부채계_연결',
 '비유동부채계_연결',
 '매입채무_연결',
 '자본총계_연결',
 '자본금_연결',
 '지배기업소유주지분_연결',
 '수익_연결',
 '총수익_연결',
 '매출원가_연결',
 '매출총이익손실_연결',
 '판매비와관리비_연결',
 '영업이익손실_연결',
 '이자수익_연결',
 '이자비용_연결',
 '법인세비용차감전계속영업이익손실_연결',
 '당기순이익지배기업지분_연결',
 'EBITDA',
 '영업활동으로인한현금흐름_연결',
 '유형자산_투자부동산감가상각비_연결',
 '무형자산상각비_연결',
 '유무형리스자산의증가_연결',
 '기말종가',
 '기말종가수정',
 '기말시가총액',
 '기말시가총액_보통주우선주',
 '평균시가총액',
 '평균시가총액_보통주우선주',
 '상장주식수_보통주',
 '기말발행주식수_보통주',
 '변동성',
 '기부금_원',
 '기부금_출처',
 '평균근속월수',
 '연간급여총액',
 '종업원평

# Modeling

## Machine Learning

In [ ]:
results_ML, scores_ML = modeling_ML(
    algos=ALGO_NAMES_ML,
    X_train=X_train, Y_train=Y_train, X_test=X_test, Y_test=Y_test,
    task=TASK,
    X_colname=X_colname,
    params=None, # 특정 알고리즘
    tune_trials=TUNE_TRIALS_ML, # 최적화 여부
    cv_splits=CV_SPLITS,
    tune_metric=TUNE_METRIC_ML,
    X_top_display=X_TOP_DISPLAY, 
    label_list=LABEL_LIST,
    random_state=RANDOM_STATE,
)


## Deep Learning

In [ ]:
# DL Modeling
results_DL, scores_DL = modeling_DL(
    algos=ALGO_NAMES_DL,
    X_train=X_train, Y_train=Y_train, X_test=X_test, Y_test=Y_test,
    task=TASK,
    class_weight_method=CLASS_WEIGHT_METHOD,
    node_list=None, # 특정 알고리즘
    validation_split=VALIDATION_SPLIT,
    validation_data=VALIDATION_DATA,
    tune_trials=TUNE_TRIALS_DL, # 최적화 여부
    cv_splits=CV_SPLITS,
    tune_metric=TUNE_METRIC_DL,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    earlystop_patient=None,
    label_list=LABEL_LIST,
    random_state=RANDOM_STATE,
)


## Performance

### Summary

In [ ]:
# Result Summary
folder_location = os.path.join(os.getcwd(),'Result')
df_scores = prediction_summary(
    folder_location=folder_location, 
    algonames=ALGO_NAMES_ML + ALGO_NAMES_DL,
    save_name=SAVE_NAME_PERFORMANCE,
)

### Significance Test

In [ ]:
# Statistical Test
df_significance, cv_scores = evaluation_MetricSignificance(
    scores=df_scores, 
    Y_train=Y_train, 
    X_train=X_train, 
    task=TASK, target_metric=METRIC_FINAL,
    cv_type=TEST_SIZE
)

## Best Model Prediction

In [ ]:
# Best Result
BEST_ALGO, BEST_MODEL, BEST_PARAMS, BEST_TESTSCORE, BEST_PREDIMPORTANCE = prediction_best(
    # df_scores=df_scores,
    df_scores=df_significance,
    X_train=locals().get('X_train_ablation', X_train), Y_train=Y_train, 
    X_test=locals().get('X_test_ablation', X_test), Y_test=Y_test,
    target_metric=METRIC_FINAL, task=TASK,
    folder_name=locals().get('model_folder', 'Model')
)


## Subgroup Robustness

In [ ]:
target_group = 'num__fe_Scope1_2_매출집약도'
# groups = {f"{target_group}_{val}": val for val in sorted(X_test[target_group].dropna().unique())}
groups = {
    '1': lambda x: x < 0,
    '2': lambda x: x>= 0
}
key_features=None

df_fairness = evaluation_reg_GroupFairness(
    best_model=BEST_MODEL,
    target_col=target_group,
    groups_dict=groups,
    X_test=X_test, Y_test=Y_test, X_colname=X_colname,
)

# Modeling with Feature Ablation

## Learning + Significance Test

In [ ]:
len(model_configs['Model A+B+C+D (Financial + Carbon + ESG + Macro)'])

In [ ]:
# (A < A+D \le A+B, A+C < A+B+C < A+B+C+D)
# Feature Blocks
def sanitize_col(col):
    c = str(col).replace(' ', '_')
    return re.sub(r'[",\[\]{}:<>\'\\]', '_', c)

FEATURE_BLOCKS_CLEAN = {
    b_name: [sanitize_col(c) for c in cols]
    for b_name, cols in FEATURE_BLOCKS.items()
}
model_configs = {
    'Model A (Financial)': FEATURE_BLOCKS_CLEAN['Block1_Financial_Control'],
    'Model A+B (Financial + Carbon)': (
         FEATURE_BLOCKS_CLEAN['Block1_Financial_Control']
         + FEATURE_BLOCKS_CLEAN['Block2_Carbon_Risk']
    ),
    'Model A+C (Financial + ESG)': (
         FEATURE_BLOCKS_CLEAN['Block1_Financial_Control']
         + FEATURE_BLOCKS_CLEAN['Block3_Social_Governance']
    ),
    'Model A+D (Financial + Macro)': (
         FEATURE_BLOCKS_CLEAN['Block1_Financial_Control']
         + FEATURE_BLOCKS_CLEAN['Block4_Industry_Macro']
    ),
    'Model A+B+C (Financial + Carbon + ESG)': (
         FEATURE_BLOCKS_CLEAN['Block1_Financial_Control']
         + FEATURE_BLOCKS_CLEAN['Block2_Carbon_Risk']
         + FEATURE_BLOCKS_CLEAN['Block3_Social_Governance']
    ),
    'Model A+B+C+D (Financial + Carbon + ESG + Macro)': (
         FEATURE_BLOCKS_CLEAN['Block1_Financial_Control']
         + FEATURE_BLOCKS_CLEAN['Block2_Carbon_Risk']
         + FEATURE_BLOCKS_CLEAN['Block3_Social_Governance']
         + FEATURE_BLOCKS_CLEAN['Block4_Industry_Macro']
    ),
}

In [ ]:
# Feature Ablation Statistical Test
df_significance_comparison, model_execution_results = evaluation_MetricSignificanceFeatureAblation(
    model_configs=model_configs,
    Y_train=Y_train, Y_test=Y_test, X_train_full=X_train, X_test_full=X_test, task=TASK,
    test_size=TEST_SIZE, target_metric=METRIC_FINAL,
    algo_names_ml=ALGO_NAMES_ML,
    algo_names_dl=ALGO_NAMES_DL,
    cv_splits=CV_SPLITS,
    tune_trials_ml=TUNE_TRIALS_ML, tune_metric_ml=TUNE_METRIC_ML, tune_trials_dl=TUNE_TRIALS_DL, tune_metric_dl=TUNE_METRIC_DL,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT, validation_data=VALIDATION_DATA,
    class_weight_method=CLASS_WEIGHT_METHOD, label_list=LABEL_LIST,
    random_state=RANDOM_STATE,
    force_rerun=False, # True 설정 시 기존 캐시 무시하고 전체 재실행
)

In [ ]:
# Best Result Selection
BEST_BLOCK = 'Model A+B+C+D (Financial + Carbon + ESG + Macro)'
df_scores = model_execution_results[BEST_BLOCK]['scores']
df_significance = model_execution_results[BEST_BLOCK]['significance']
X_train_ablation = X_train[model_execution_results[BEST_BLOCK]['features']]
X_test_ablation = X_test[model_execution_results[BEST_BLOCK]['features']]
model_folder = model_execution_results[BEST_BLOCK]['model_dir']

## Performance

### Best Model Prediction

In [ ]:
# Best Result
BEST_ALGO, BEST_MODEL, BEST_PARAMS, BEST_TESTSCORE, BEST_PREDIMPORTANCE = prediction_best(
    df_scores=df_scores,
    # df_scores=df_significance,
    X_train=locals().get('X_train_ablation', X_train), Y_train=Y_train, 
    X_test=locals().get('X_test_ablation', X_test), Y_test=Y_test,
    target_metric=METRIC_FINAL, task=TASK,
    folder_name=locals().get('model_folder', 'Model')
)


### Subgroup Robustness

In [ ]:
target_group = 'num__fe_Scope1_2_매출집약도'
# groups = {f"{target_group}_{val}": val for val in sorted(X_test[target_group].dropna().unique())}
groups = {
    '1': lambda x: x < 0,
    '2': lambda x: x>= 0
}
key_features=None

df_fairness = evaluation_reg_GroupFairness(
    best_model=BEST_MODEL,
    target_col=target_group,
    groups_dict=groups,
    X_test=X_test, Y_test=Y_test, X_colname=X_colname,
)

# XAI

## Explanation

In [ ]:
# Rename of Columns
X_selected = None
# df_domain_rename = pd.read_excel(os.path.join(FOLDER_GDM, '[Rename_20251228].xlsx'), sheet_name='Features_SI')
# df_domain_rename = {i[0]:i[1] for i in df_domain_rename[['변수설명(한글)', '변수설명(영어)']].values}
# if LANGUAGE == 'kr':
#     X_selected = [col for col in X_colname if col in list(df_domain_rename.keys())]
# else:
#     X_selected = [df_rename[col] for col in X_colname if col in list(df_domain_rename.keys())]

# SHAP 실행
explanation_SHAP(
    model=BEST_MODEL, 
    X_train=locals().get('X_train_ablation', X_train), Y_train=Y_train, 
    X_test=locals().get('X_test_ablation', X_test), Y_test=Y_test,
    X_colname=X_colname, X_selected=X_selected,
    interaction=INTERACTION,
    interaction_samples=INTERACTION_SAMPLES,
    interaction_batch=INTERACTION_BATCH,
    max_display=X_TOP_DISPLAY,
)


## Subgroup Comparison

In [ ]:
# # 연가구소득
# target_group = 'num__fe_Scope1_2_매출집약도'
# # groups = {f"{target_group}_{val}": val for val in sorted(X_test[target_group].dropna().unique())}
# groups = {
#     '1': lambda x: x < 0,
#     '2': lambda x: x>= 0
# }
# key_features=None

# df_comparison = explanation_SHAP_groupcomparing(
#     target_col=target_group,
#     groups_dict=groups,
#     X_train=X_train, X_test=X_test,
#     X_colname=X_colname,
#     # scaler_dict=BEST_PARAMS['scaler_type'],
#     best_algo=BEST_ALGO,
#     max_display=10,
#     save_csv=True,
#     target_features=key_features
# )

# Causality Analysis

In [ ]:
model_best, df_result, df_sensitivity = modeling_CF(
    df=df_prep,
    Y_colname=Y_colname,
    T_colname=T_colname,
    X_covariate=X_covariate,
    cv_candidate=CV_CANDIDATE,
    n_estimators=N_ESTIMATORS,      # 리뷰어 피드백 시 변경 가능
    cv_splits=CV_SPLITS,            # 강건성 확인 시 변경 가능
    random_state=RANDOM_STATE,
)